# AI in Agriculture: Crop Yield Prediction & Food Security Analytics
## Africa & USA — Exploratory Analysis, Visualisation & ML

This notebook explores the feature-engineered agriculture dataset built by the ETL pipeline, visualises key patterns, and trains a simple linear regression model to predict cereal yield.

**Data:** `../data/processed/agriculture_with_features.csv`  
**Source:** World Bank Open Data API.

## 1. Setup & Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
print('Libraries imported successfully.')

## 2. Load Data

In [ ]:
DATA_PATH = Path('../data/processed/agriculture_with_features.csv')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nRegions:', df['region'].unique())
print('Countries:', df['country_name'].nunique())
print('Year range:', df['year'].min(), '-', df['year'].max())

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing values per column
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'missing_pct': missing_pct})

## 4. Visualisation 1 — Average Cereal Yield by Country

In [ ]:
yield_by_country = (df.groupby(['country_name', 'region'])['cereal_yield_kg_ha']
                      .mean().reset_index()
                      .sort_values('cereal_yield_kg_ha', ascending=False))
fig = px.bar(yield_by_country, x='country_name', y='cereal_yield_kg_ha',
             color='region', title='Average Cereal Yield by Country (kg/hectare)',
             labels={'cereal_yield_kg_ha': 'Cereal yield (kg/ha)',
                     'country_name': 'Country'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**Insight:** The USA sits far above every African country in cereal yield, illustrating the productivity gap that AI-driven precision agriculture aims to close. Within Africa, a handful of countries (notably Egypt and South Africa) lead the pack thanks to irrigation and higher input use.

## 5. Visualisation 2 — Cereal Yield Trend Over Time

In [ ]:
top_countries = (df.groupby('country_name')['cereal_yield_kg_ha'].mean()
                   .sort_values(ascending=False).head(6).index.tolist())
trend = df[df['country_name'].isin(top_countries)]
fig = px.line(trend.sort_values('year'), x='year', y='cereal_yield_kg_ha',
              color='country_name', markers=True,
              title='Cereal Yield Trend Over Time (Top 6 Countries)',
              labels={'cereal_yield_kg_ha': 'Cereal yield (kg/ha)', 'year': 'Year'})
fig.show()

**Insight:** Yield trajectories are uneven — some countries show steady improvement while others plateau or fluctuate year to year. This country-specific volatility means AI forecasting models must be tuned per country rather than assuming one continental trend.

## 6. Visualisation 3 — Fertilizer vs Cereal Yield

In [ ]:
scatter_df = df.dropna(subset=['fertilizer_kg_ha', 'cereal_yield_kg_ha',
                                'food_production_index'])
fig = px.scatter(scatter_df, x='fertilizer_kg_ha', y='cereal_yield_kg_ha',
                 color='region', size='food_production_index',
                 hover_name='country_name',
                 title='Fertilizer Consumption vs Cereal Yield',
                 labels={'fertilizer_kg_ha': 'Fertilizer (kg/ha)',
                         'cereal_yield_kg_ha': 'Cereal yield (kg/ha)'})
fig.show()

**Insight:** There is a clear positive relationship between fertilizer intensity and cereal yield. The USA occupies the high-input / high-output corner, while most African countries cluster at low fertilizer use — highlighting where smarter, AI-optimised input allocation could raise output.

## 7. Visualisation 4 — Correlation Heatmap

In [ ]:
num_cols = ['cereal_yield_kg_ha', 'agr_land_pct', 'food_production_index',
            'fertilizer_kg_ha', 'agr_value_added_pct', 'rural_population_pct',
            'yield_growth_rate', 'fertilizer_efficiency', 'food_security_score']
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix of Agricultural Indicators')
plt.tight_layout()
plt.show()

**Insight:** Fertilizer consumption and cereal yield are strongly correlated, and both feed the food-security score. Rural population share correlates negatively with yield and food security, confirming that highly rural economies face greater food-security pressure.

## 8. Visualisation 5 — AI Investment Priority Distribution

In [ ]:
priority = (df.groupby('country_name')
              .agg(ai_investment_priority=('ai_investment_priority', 'first'),
                   food_security_score=('food_security_score', 'mean'))
              .reset_index())
order = ['Critical', 'High', 'Medium', 'Low']
priority['ai_investment_priority'] = pd.Categorical(
    priority['ai_investment_priority'], categories=order, ordered=True)
priority = priority.sort_values(['ai_investment_priority', 'food_security_score'])
fig = px.bar(priority, x='country_name', y='food_security_score',
             color='ai_investment_priority',
             category_orders={'ai_investment_priority': order},
             title='Food Security Score & AI Investment Priority by Country',
             labels={'food_security_score': 'Avg food security score',
                     'country_name': 'Country',
                     'ai_investment_priority': 'AI priority'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**Insight:** The 'Critical' and 'High' priority buckets group the lowest-scoring countries — the clearest targets for AI-driven agricultural investment. This ranking turns raw indicators into an actionable prioritisation for policymakers and investors.

## 9. Visualisation 6 — Africa vs USA Comparison

In [ ]:
metrics = ['cereal_yield_kg_ha', 'fertilizer_kg_ha', 'food_production_index',
           'food_security_score']
region_avg = df.groupby('region')[metrics].mean().reset_index()
melted = region_avg.melt(id_vars='region', var_name='metric', value_name='value')
fig = px.bar(melted, x='metric', y='value', color='region', barmode='group',
             title='Africa vs USA — Average Agricultural Metrics',
             labels={'value': 'Average value', 'metric': 'Metric'})
fig.update_layout(xaxis_tickangle=-20)
fig.show()

**Insight:** Side by side, the USA dwarfs the African average on cereal yield and fertilizer use, yet food production index (a relative measure) is closer — showing African output is growing from a low base. The composite food-security score quantifies the remaining gap.

## 10. Machine Learning — Predicting Cereal Yield

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

ml_df = df.dropna(subset=['cereal_yield_kg_ha', 'fertilizer_kg_ha', 'agr_land_pct'])
X = ml_df[['fertilizer_kg_ha', 'agr_land_pct']]
y = ml_df['cereal_yield_kg_ha']
print(f'Training on {len(ml_df)} complete records')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'R2 score: {r2_score(y_test, y_pred):.3f}')
print(f'MAE: {mean_absolute_error(y_test, y_pred):,.1f} kg/ha')
print('\nCoefficients:')
for feat, coef in zip(X.columns, model.coef_):
    print(f'  {feat}: {coef:,.2f}')
print(f'  intercept: {model.intercept_:,.2f}')

**Insight:** Even a simple two-feature linear model captures a meaningful share of yield variation, with fertilizer consumption as the dominant positive driver. This confirms that input intensity is a key, learnable predictor — a foundation for richer AI yield-forecasting models.

## 11. Conclusion — Key Findings

1. **Productivity gap is large but explainable.** USA cereal yields far exceed the African cohort, and the gap tracks closely with fertilizer intensity — a lever AI-guided precision agriculture can pull.

2. **Fertilizer is the strongest predictor of yield.** Both the correlation heatmap and the regression model identify fertilizer consumption as the dominant driver of cereal yield.

3. **Food-security risk is concentrated.** The composite food-security score and AI-investment-priority ranking cleanly isolate a 'Critical' cohort of countries needing the most support.

4. **High rural dependence coincides with lower food security.** Countries with large rural populations tend to score lower, making them priority targets for rural-focused AI interventions.

5. **Yield growth is country-specific and volatile.** Year-over-year trajectories differ markedly across countries, so AI forecasting must model each country individually rather than assume one continental trend.